In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

True

Install Libraries

In [3]:
!pip install -q \
youtube-transcript-api \
langchain \
langchain-community \
langchain-core \
langchain-groq \
langchain-text-splitters \
sentence-transformers \
faiss-cpu \
rank-bm25 \
ragas \
langsmith

In [4]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

C:\Users\Anshi\AppData\Local\Temp\ipykernel_13404\819515520.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Step 1a - Indexing (Document Ingestion)

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_id = "Gfr50f6ZBvo"

try:
    api = YouTubeTranscriptApi()

    transcript_list = api.fetch(video_id, languages=["en"])

    transcript_with_time = []

    for chunk in transcript_list:
        transcript_with_time.append(
            f"[{chunk.start:.2f}s] {chunk.text}"
        )

    transcript = " ".join(transcript_with_time)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

[0.08s] the following is a conversation with [1.76s] demus hasabis [3.52s] ceo and co-founder of deepmind [6.72s] a company that has published and builds [8.64s] some of the most incredible artificial [11.20s] intelligence systems in the history of [13.20s] computing including alfred zero that [16.00s] learned [16.88s] all by itself to play the game of gold [18.96s] better than any human in the world and [21.44s] alpha fold two that solved protein [24.56s] folding [25.68s] both tasks considered nearly impossible [28.72s] for a very long time [31.12s] demus is widely considered to be one of [33.12s] the most brilliant and impactful humans [35.60s] in the history of artificial [37.12s] intelligence and science and engineering [39.84s] in general [41.12s] this was truly an honor and a pleasure [43.76s] for me to finally sit down with him for [46.08s] this conversation and i'm sure we will [48.56s] talk many times again in the future [51.44s] this is the lex friedman podcast to [53.28s] su

In [7]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='the following is a conversation with', start=0.08, duration=3.44), FetchedTranscriptSnippet(text='demus hasabis', start=1.76, duration=4.96), FetchedTranscriptSnippet(text='ceo and co-founder of deepmind', start=3.52, duration=5.119), FetchedTranscriptSnippet(text='a company that has published and builds', start=6.72, duration=4.48), FetchedTranscriptSnippet(text='some of the most incredible artificial', start=8.639, duration=4.561), FetchedTranscriptSnippet(text='intelligence systems in the history of', start=11.2, duration=4.8), FetchedTranscriptSnippet(text='computing including alfred zero that', start=13.2, duration=3.68), FetchedTranscriptSnippet(text='learned', start=16.0, duration=2.96), FetchedTranscriptSnippet(text='all by itself to play the game of gold', start=16.88, duration=4.559), FetchedTranscriptSnippet(text='better than any human in the world and', start=18.96, duration=5.6), FetchedTranscriptSnippet(text='alph

Step 1b - Indexing(Text Splitting)

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [9]:
len(chunks)

219

In [10]:
chunks[100]

Document(metadata={}, page_content='and then [3651.04s] only at the last stage validate it in [3652.96s] the wet lab so you could you know in [3655.20s] terms of the search space of discovering [3656.88s] new drugs you know it takes 10 years [3658.72s] roughly to go from uh uh to to go from [3661.36s] uh you know identifying a target to uh [3664.56s] having a drug candidate um maybe that [3666.72s] could be shortened to you know by an [3668.16s] order of magnitude with if you could do [3670.24s] most of that that that work in silico so [3673.44s] in order to get to a virtual cell [3675.60s] we have to build up uh uh understanding [3678.24s] of different parts of biology and the [3679.60s] interactions and and um so you know [3682.16s] every every few years we talk about this [3684.00s] with i talked about this with paul and [3685.52s] then finally last year after alpha fault [3687.68s] i said now is the time we can finally go [3690.00s] for it and and alpha falls the first [3691.68s] p

Step 1c & 1d - Indexing(Embedding Generation and Storing in Vector Store)

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_KEY") is not None)

True


In [12]:
# Groq LLM
from langchain_groq import ChatGroq
llm = ChatGroq(
    model_name="llama-3.1-8b-instant"
)

In [13]:
# For embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

C:\Users\Anshi\AppData\Local\Temp\ipykernel_13404\2310780361.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [14]:
vector_store = FAISS.from_documents(chunks,embeddings)

In [15]:
vector_store.index_to_docstore_id

{0: '5e316851-9f4a-4dfb-98dd-1e9ad60b26d0',
 1: '885a07dc-d09f-4c60-ac6d-4b0aea6b0f11',
 2: 'bcf65cb3-dd02-4afd-8577-c511f235dee9',
 3: 'e91ea36b-26a3-47b2-942b-2d0bbcff3b4d',
 4: '4f5fd8c8-afb5-456e-bf99-cd25a20d4b9e',
 5: '2d3ca212-5039-404a-8379-90506e63c9f5',
 6: '42459c2e-5fc8-456a-b424-96af8481c679',
 7: '765d9bbb-2667-4746-96f9-e4a6744cf8d4',
 8: 'c26be0ac-9adb-42b8-ac5a-556f448129ce',
 9: 'd1669bba-5c43-4a6b-ab65-a14c2eb67aed',
 10: '7c28bc8f-68c8-44b9-b049-6943b24468ca',
 11: '6f855aa8-72ef-4c7f-8268-799caaf5886d',
 12: '61e64112-fa74-4650-9265-941ef0f7d84a',
 13: '3603f8fb-ec5e-40dd-ba92-19d67a234957',
 14: 'e66c29fb-4f2b-4efd-ab22-c5208ac9e526',
 15: '5e26250f-c771-44a6-8e29-54df032ec6ab',
 16: '9be5398b-3bba-4e71-bd65-fcba239c878a',
 17: 'e04bda90-4bc0-4b14-a330-483c161c30b3',
 18: '0cc3f596-0995-45c2-908a-4cac8992b77b',
 19: 'e52bc3d3-c172-4b2b-a45e-3558c6658bd1',
 20: '093b8f9f-9aa2-4813-b575-115888549c6f',
 21: '2299edf6-2dee-449a-8767-30890d754a40',
 22: 'b129f599-9a30-

In [16]:
vector_store.get_by_ids(['9ed96330-2d51-417f-bd23-57e37e770f07'])

[]

Step 2 - Retrieval

In [17]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs = {"k": 4})

In [18]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D808B57610>, search_kwargs={'k': 4})

In [19]:
retriever.invoke('what is the video about?') #retriver is a runnable object, so we can call it like a function. It will return the top k documents that are most similar to the query.

[Document(id='e0651399-94c6-4672-a177-1531281cc2ce', metadata={}, page_content="fun to play for prolonged [7655.52s] periods of time [7657.60s] and it does seem like this puzzle like [7659.52s] you mentioned the more you learn about [7661.68s] it the more you realize how little you [7663.84s] know [7664.80s] so it humbles you but excites you by the [7667.04s] possibility of learning more it's one [7669.44s] heck of a one heck of a puzzle we got [7671.84s] going on here um so like i mentioned of [7674.96s] all the people in the world you're very [7677.04s] likely to be the one who creates the agi [7680.56s] system [7681.92s] um that achieves human level [7684.24s] intelligence and goes beyond it so if [7686.40s] you got a chance and very well you could [7688.64s] be the person that goes into the room [7690.16s] with the system and have a conversation [7692.96s] maybe you only get to ask one question [7695.20s] if you do [7696.48s] what question would you ask her [7699.28s] i would proba

Step 3 - Augmentation

In [20]:
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0.2,
    streaming=True
)  

In [21]:
prompt = PromptTemplate(
    template = """
    You are a helpful assistant.
    Answer ONLY from the provided transcript context. If you don't know the answer, just say you don't know.
    
    {context}
    Question: {question}
    """,
    input_variables = ["context", "question"]
)

In [22]:
question = "Is the topic of Khalibali discussed in this video? If yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [23]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"things right than the standard [7771.04s] model of physics which we know doesn't [7772.72s] work but we still keep adding to so um [7776.16s] and and that's how i think the beginning [7777.76s] of an explanation would look and it [7779.04s] would start encompassing many of the [7780.64s] mysteries that we have wondered about [7782.24s] for thousands of years like you know [7784.24s] consciousness [7785.76s] uh life and gravity all of these things [7788.72s] yeah giving us a glimpses of [7790.32s] explanations for those things yeah [7792.48s] well um [7793.92s] damas dear one of the special [7796.40s] human beings in this giant puzzle of [7798.24s] ours and it's a huge honor that you [7800.16s] would take a pause from the bigger [7801.44s] puzzle to solve this small puzzle of a [7803.28s] conversation with me today it's truly an [7805.28s] honor and a pleasure thank you thank you [7806.64s] i really enjoyed it thanks lex [7808.96s] thanks for listening to this [7809.92s] conversation\n

In [24]:
final_prompt = prompt.invoke({"context" : context_text, "question" : question})

In [25]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context. If you don't know the answer, just say you don't know.\n    \n    things right than the standard [7771.04s] model of physics which we know doesn't [7772.72s] work but we still keep adding to so um [7776.16s] and and that's how i think the beginning [7777.76s] of an explanation would look and it [7779.04s] would start encompassing many of the [7780.64s] mysteries that we have wondered about [7782.24s] for thousands of years like you know [7784.24s] consciousness [7785.76s] uh life and gravity all of these things [7788.72s] yeah giving us a glimpses of [7790.32s] explanations for those things yeah [7792.48s] well um [7793.92s] damas dear one of the special [7796.40s] human beings in this giant puzzle of [7798.24s] ours and it's a huge honor that you [7800.16s] would take a pause from the bigger [7801.44s] puzzle to solve this small puzzle of a [7803.28s] conversation with me 

Step 4 - Generation

In [26]:
answer = llm.invoke(final_prompt)
print(answer.content)

I don't know.


Building Chain

In [27]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [28]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [29]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [30]:
parallel_chain.invoke('who is demis')

{'context': "[43.76s] for me to finally sit down with him for [46.08s] this conversation and i'm sure we will [48.56s] talk many times again in the future [51.44s] this is the lex friedman podcast to [53.28s] support it please check out our sponsors [55.44s] in the description and now dear friends [58.24s] here's demis [59.60s] hassabis [61.44s] let's start with a bit of a personal [62.96s] question [63.92s] am i an ai program you wrote to [66.80s] interview people until i get good enough [68.72s] to interview you [70.96s] well i'll be impressed if if you were [72.96s] i'd be impressed by myself if you were i [74.80s] don't think we're quite up to that yet [76.40s] but uh maybe you're from the future lex [78.72s] if you did would you tell me is that is [80.88s] that a good thing to tell a language [82.56s] model that's tasked with interviewing [84.96s] that it is in fact um ai maybe we're in [87.76s] a kind of meta turing test uh probably [90.24s] probably it would be a good idea not t

In [31]:
parser = StrOutputParser()

In [32]:
main_chain = parallel_chain | prompt | llm | parser

In [33]:
response = main_chain.invoke(
    "Can you summarize the video?"
)

print(response)

The video is a Lex Fridman podcast episode featuring Demis Hassabis, the co‑founder of DeepMind. In the conversation, they touch on several high‑level topics:

* **AI and the future of interviewing** – Hassabis jokes about an AI program that could interview people, and the idea of a “meta Turing test” where an AI might be asked to identify itself as a language model.
* **The nature of reality** – Hassabis muses about what the true nature of reality might be, a question he would ask an AGI if he could meet it.
* **Human‑AI interaction** – They discuss the humbling realization that what we once thought impossible is becoming possible, and how AI systems can influence millions of people.
* **The role of AI in games** – Hassabis reflects on how AI has transformed games like Go, and the broader impact of AI on society.
* **Closing remarks** – Lex thanks Hassabis, and the episode ends with a quote from Edskar Dykstra about computer science.

Overall, the episode blends humor, philosophical s